# The one cortex

Built from commit `5ae730b`. One recurrent network hears conversation and reading as a stream, predicts it,
remembers it in a fast hippocampal memory, and speaks from the same tissue - learning by incremental predictive
coding. See `PREREGISTRATION_onecortex.md`.

In [ ]:
import os, sys, time, glob, subprocess, json
W = "/kaggle/working/brain"
for d in ("brain", "tests"):
    os.makedirs(os.path.join(W, d), exist_ok=True)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
WIKI = os.path.dirname(glob.glob("/kaggle/input/**/wikitext-103-raw/wiki.train.raw", recursive=True)[0])
TALK = os.path.dirname([p for p in glob.glob("/kaggle/input/**/train.csv", recursive=True) if "dialog" in p.lower()][0])
print(WIKI, TALK)

In [ ]:
%%writefile /kaggle/working/brain/brain/__init__.py
"""A conversational brain: lexical cortex, prefrontal working memory, hippocampal map and episodic store, a
basal-ganglia gate that learns which of them answers, and sleep. No pretrained weights, no gradients."""

In [ ]:
%%writefile /kaggle/working/brain/brain/onecortex.py
"""One cortex: a recurrent language network that hears a conversation as a stream, predicts it, remembers it,
and speaks from the same tissue - learning by a local rule throughout.

Why this replaces the pipeline. The self-trained brain chose among candidates it was handed (a stretch of a page,
a line someone once said) using features written by hand, one sentence at a time, behind hand-written routing.
On open conversation it matched words instead of meaning ("yo" -> "yo-yo"), could not say anything new, forgot
what was said two turns ago, and stored "you're a terrible chatbot" as a fact. None of that is a training
problem. This file is the architecture that could fix it, built from pieces that have each earned a place:

  cortex        Two recurrent layers. Each layer's activity is predicted from the layer below *and from its own
                activity a moment ago* (temporal predictive coding; Millidge et al. 2023), so the state carries
                the conversation - "that", the topic, the joke setup - without any rule deciding what to keep.
  learning      Incremental predictive coding (Salvatori et al. 2024): at every word the activities settle to
                reduce prediction error, and at every settling step each synapse changes by its own layer's
                error times its own input. No backpropagation, and none through time: the previous state is
                treated as given, as it is for a neuron. In run 3 of the deep cortex this rule improved at every
                checkpoint while ordinary predictive coding drifted.
  hippocampus   A fast associative memory inside the stream: a delta-rule matrix that binds each moment's key to
                what came next, decays slowly, and is wiped between conversations. The top layer's prediction
                reads it. It learns within the conversation, one write per word - the mechanism that gave the only
                positive result in the MICrONS run (carrying a fast memory across a stream: +0.37 bits/char).
                Keys and values come through fixed random projections, as dentate gyrus inputs are fixed.
  Broca         The same cortex run forward: given the conversation so far and a speaker mark, it samples words
                until the turn ends. Several drafts are sampled; which one is said is a choice.

Tokens: lowercase words plus <unk>, <a>/<b> (who is speaking) and <p> (a paragraph boundary in read text).
"""
import math
import random
import re
import time

import torch
import torch.nn.functional as Fn

SPECIAL = ["<unk>", "<a>", "<b>", "<p>"]
UNK, SPK_A, SPK_B, PARA = 0, 1, 2, 3


# ------------------------------------------------------------------------------------------------ text
def words(text):
    return re.findall(r"[a-z0-9]+(?:'[a-z]+)?|[.,!?;:'\"()-]", text.lower())


class Vocab:
    def __init__(self, counts, size):
        keep = [w for w, _ in counts.most_common(size) if w not in SPECIAL][: size - len(SPECIAL)]
        self.itos = SPECIAL + keep
        self.stoi = {w: i for i, w in enumerate(self.itos)}

    def __len__(self):
        return len(self.itos)

    def ids(self, text):
        return [self.stoi.get(w, UNK) for w in words(text)]

    def dialogue(self, turns, first=SPK_A):
        """A conversation as one stream: speaker mark, then the turn, alternating."""
        out = []
        for i, turn in enumerate(turns):
            out.append(first if i % 2 == 0 else (SPK_B if first == SPK_A else SPK_A))
            out += self.ids(turn)
        return out

    def text(self, ids):
        out = []
        for i in ids:
            w = self.itos[i]
            if w in ("<a>", "<b>", "<p>"):
                continue
            out.append(w)
        s = " ".join(out)
        s = re.sub(r" ([.,!?;:)])", r"\1", s)
        s = re.sub(r"\( ", "(", s)
        s = re.sub(r" ' ", "'", s)
        return s


# ------------------------------------------------------------------------------------------------ the cortex
class OneCortex(torch.nn.Module):
    """Two recurrent layers of leaky neurons, a fast memory, and a readout.

    Each neuron has a membrane potential that carries a fraction `leak` of itself into the next word and takes the
    rest from its inputs. Higher areas integrate over longer timescales than lower ones (Honey et al. 2012;
    Chaudhuri et al. 2015), so layer 2 is slower than layer 1. With leak 0 this is exactly the run-1 cortex.
    """

    def __init__(self, vocab, embed=128, hidden=512, mem=64, memory=True, seed=0, leak=(0.0, 0.0)):
        super().__init__()
        g = torch.Generator().manual_seed(seed)
        self.V, self.d, self.H, self.m, self.memory = vocab, embed, hidden, mem, memory
        self.a1, self.a2 = float(leak[0]), float(leak[1])

        def w(o, i):
            return torch.nn.Parameter(torch.randn(o, i, generator=g) / math.sqrt(i))

        self.E = torch.nn.Parameter(torch.randn(vocab, embed, generator=g) * 0.1)
        self.W1, self.U1, self.b1 = w(hidden, embed), w(hidden, hidden), torch.nn.Parameter(torch.zeros(hidden))
        self.W2, self.U2, self.b2 = w(hidden, hidden), w(hidden, hidden), torch.nn.Parameter(torch.zeros(hidden))
        self.Wm = w(hidden, mem)                                  # what the hippocampal recall contributes
        self.Wo, self.bo = w(vocab, hidden), torch.nn.Parameter(torch.zeros(vocab))
        with torch.no_grad():                                     # recurrence starts gentle, so state neither dies
            self.U1.mul_(0.5)                                     # nor explodes before it has learned anything
            self.U2.mul_(0.5)
        # fixed random projections into and out of the fast memory: wiring, not learning
        self.register_buffer("Pk", torch.randn(mem, hidden, generator=g) / math.sqrt(hidden))
        self.register_buffer("Pv", torch.randn(mem, hidden, generator=g) / math.sqrt(hidden))
        self.decay, self.write = 0.98, 0.5

    @staticmethod
    def f(x):
        return torch.tanh(x)

    @staticmethod
    def df(x):
        return 1.0 - torch.tanh(x) ** 2

    # ---------------------------------------------------------------- state
    def fresh(self, n, device):
        """A quiet cortex: potentials at rest, an empty memory, and every synapse's eligibility trace at zero."""
        zH = torch.zeros(n, self.H, device=device)
        return {"v1": zH.clone(), "v2": zH.clone(), "M": torch.zeros(n, self.m, self.m, device=device),
                "k": torch.zeros(n, self.m, device=device),
                # eligibility traces: each synapse's fading memory of its own presynaptic input. In a layer every
                # neuron leaks at the same rate, so a trace depends only on the presynaptic side and is shared by
                # all of that input's synapses - one vector per input, not one number per synapse.
                "zW1": torch.zeros(n, self.d, device=device), "zU1": zH.clone(), "zb1": torch.zeros(n, 1, device=device),
                "zW2": zH.clone(), "zU2": zH.clone(), "zm": torch.zeros(n, self.m, device=device),
                "zb2": torch.zeros(n, 1, device=device)}

    def reset(self, state, rows):
        """Forget the given streams: a new conversation starts at rest, with an empty memory and no traces."""
        if rows is None or not bool(rows.any()):
            return state
        keep = (~rows).float()
        out = {}
        for k, v in state.items():
            out[k] = v * keep.view(-1, *([1] * (v.dim() - 1)))
        return out

    def recall(self, state):
        if not self.memory:
            return torch.zeros_like(state["k"])
        return torch.bmm(state["M"], state["k"][:, :, None])[:, :, 0]

    def drive1(self, e, state):
        return Fn.linear(e, self.W1, self.b1) + Fn.linear(self.f(state["v1"]), self.U1)

    def drive2(self, h1, state, r):
        return (Fn.linear(self.f(h1), self.W2, self.b2) + Fn.linear(self.f(state["v2"]), self.U2)
                + Fn.linear(r, self.Wm))

    def predict(self, e, state, r, h1=None):
        """Each layer's predicted potential: what it carries over by leak, plus what its inputs now drive."""
        mu1 = self.a1 * state["v1"] + (1 - self.a1) * self.drive1(e, state)
        h1 = mu1 if h1 is None else h1
        mu2 = self.a2 * state["v2"] + (1 - self.a2) * self.drive2(h1, state, r)
        return mu1, mu2

    @torch.no_grad()
    def traces_now(self, e, state, h1, r):
        """The traces as they stand this moment: last moment's, decayed by the neuron's leak, plus this input."""
        a1, a2 = self.a1, self.a2
        return {"zW1": a1 * state["zW1"] + (1 - a1) * e,
                "zU1": a1 * state["zU1"] + (1 - a1) * self.f(state["v1"]),
                "zb1": a1 * state["zb1"] + (1 - a1),
                "zW2": a2 * state["zW2"] + (1 - a2) * self.f(h1),
                "zU2": a2 * state["zU2"] + (1 - a2) * self.f(state["v2"]),
                "zm": a2 * state["zm"] + (1 - a2) * r,
                "zb2": a2 * state["zb2"] + (1 - a2)}

    @torch.no_grad()
    def advance(self, state, h1, h2, e=None, r=None):
        """After a word: the settled potentials become the state, the traces move on, and the hippocampus writes."""
        s2 = self.f(h2)
        new = {"v1": h1, "v2": h2}
        if e is not None:
            new.update(self.traces_now(e, state, h1, r if r is not None else self.recall(state)))
        else:
            new.update({k: state[k] for k in ("zW1", "zU1", "zb1", "zW2", "zU2", "zm", "zb2")})
        k = Fn.normalize(Fn.linear(s2, self.Pk), dim=1)
        if self.memory:
            v = Fn.linear(s2, self.Pv)
            prev = state["k"]
            got = torch.bmm(state["M"], prev[:, :, None])[:, :, 0]
            # delta rule: bind the previous moment's key to what this moment turned out to be
            M = self.decay * state["M"] + self.write * torch.bmm((v - got)[:, :, None], prev[:, None, :])
        else:
            M = state["M"]
        new.update(M=M, k=k)
        return new

    def forward_step(self, x, state):
        """Feedforward: predict the next word from this one and the state, without settling. Used to evaluate
        and to speak - with no target there is nothing to settle toward, so the prediction is the equilibrium."""
        e = self.E[x]
        r = self.recall(state)
        mu1, mu2 = self.predict(e, state, r)
        logits = Fn.linear(self.f(mu2), self.Wo, self.bo)
        return logits, mu1, mu2

    def params(self):
        return sum(p.numel() for p in self.parameters())


# ------------------------------------------------------------------------------------------------ learning
@torch.no_grad()
def local_changes(net, x, y, state, h1, h2, beta):
    """Every layer's prediction error at these potentials, the output error, and the energy per stream."""
    e = net.E[x]
    r = net.recall(state)
    mu1, mu2 = net.predict(e, state, r, h1)
    eps1, eps2 = h1 - mu1, h2 - mu2
    logits = Fn.linear(net.f(h2), net.Wo, net.bo)
    p = torch.softmax(logits, 1)
    rows = torch.arange(len(y), device=y.device)
    ce = torch.logsumexp(logits, 1) - logits[rows, y]
    p[rows, y] -= 1.0
    energy = 0.5 * ((eps1 * eps1).sum(1) + (eps2 * eps2).sum(1)) + beta * ce
    return {"eps1": eps1, "eps2": eps2, "p": p, "e": e, "r": r, "energy": energy, "ce": ce}


@torch.no_grad()
def settle(net, x, y, state, steps, rate, beta, opt=None):
    """Settle the potentials for one word; with `opt`, synapses change at every settling step (incremental PC).
    Returns the settled h1, h2 and the loss before settling."""
    e = net.E[x]
    r = net.recall(state)
    mu1, mu2 = net.predict(e, state, r)
    h1, h2 = mu1.clone(), mu2.clone()
    n = len(y)
    step = torch.full((n, 1), float(rate), device=x.device)
    cur = local_changes(net, x, y, state, h1, h2, beta)
    loss0 = float(cur["ce"].mean())
    for _ in range(steps):
        # layer 1 hears layer 2's error through W2, scaled by how much of layer 2 is driven rather than carried
        d1 = -cur["eps1"] + net.df(h1) * ((1 - net.a2) * (cur["eps2"] @ net.W2))
        d2 = -cur["eps2"] - net.df(h2) * ((beta * cur["p"]) @ net.Wo)
        t1, t2 = h1 + step * d1, h2 + step * d2
        nxt = local_changes(net, x, y, state, t1, t2, beta)
        ok = ((nxt["energy"] <= cur["energy"]) & torch.isfinite(nxt["energy"]))[:, None]
        h1, h2 = torch.where(ok, t1, h1), torch.where(ok, t2, h2)
        for k in ("eps1", "eps2", "p"):
            cur[k] = torch.where(ok, nxt[k], cur[k])
        cur["energy"] = torch.where(ok[:, 0], nxt["energy"], cur["energy"])
        step = torch.where(ok, step, step * 0.5)
        if opt is not None:
            apply(net, opt, x, state, h1, h2, cur, beta)
    return h1, h2, loss0


@torch.no_grad()
def local_grads(net, x, state, h1, h2, cur, beta):
    """Each synapse's change: its layer's error (rescaled by 1/beta) times its eligibility trace - the fading
    memory of its own input (e-prop; Bellec et al. 2020). With no leak the trace is just the current input, and
    this is exactly run 1's rule. Returned gradient-shaped (the negative of the change) for the optimiser."""
    n = len(x)
    e1, e2 = cur["eps1"] / beta, cur["eps2"] / beta
    p = cur["p"]
    z = net.traces_now(cur["e"], state, h1, cur["r"])
    s2 = net.f(h2)
    gE = torch.zeros_like(net.E)
    # the embedding is a lookup, so its synapses see only the word now; no trace is kept for them
    gE.index_add_(0, x, -((1 - net.a1) * (e1 @ net.W1)) / n)
    return {"Wo": p.T @ s2 / n, "bo": p.mean(0),
            "W2": -(e2.T @ z["zW2"]) / n, "U2": -(e2.T @ z["zU2"]) / n, "b2": -(e2 * z["zb2"]).mean(0),
            "Wm": -(e2.T @ z["zm"]) / n,
            "W1": -(e1.T @ z["zW1"]) / n, "U1": -(e1.T @ z["zU1"]) / n, "b1": -(e1 * z["zb1"]).mean(0), "E": gE}


def apply(net, opt, x, state, h1, h2, cur, beta):
    """Hand each synapse its local change and let the optimiser (per-synapse, so local) take the step."""
    g = local_grads(net, x, state, h1, h2, cur, beta)
    opt.zero_grad(set_to_none=True)
    for name, grad in g.items():
        getattr(net, name).grad = grad
    total = torch.sqrt(sum((v * v).sum() for v in g.values()))
    if not torch.isfinite(total):
        return
    if total > 1.0:
        for v in g.values():
            v.mul_(1.0 / total)
    opt.step()


def bp_chunk(net, opt, xs, ys, starts, state):
    """Backprop through the chunk (truncated BPTT): the yardstick, never part of the brain."""
    opt.zero_grad(set_to_none=True)
    loss = 0.0
    s = {k: v.detach() for k, v in state.items()}
    for t in range(xs.shape[0]):
        s = net.reset(s, starts[t])
        logits, mu1, mu2 = net.forward_step(xs[t], s)
        loss = loss + Fn.cross_entropy(logits, ys[t])
        s2 = net.f(mu2)
        k = Fn.normalize(Fn.linear(s2.detach(), net.Pk), dim=1)
        if net.memory:
            v = Fn.linear(s2.detach(), net.Pv)
            got = torch.bmm(s["M"], s["k"][:, :, None])[:, :, 0]
            M = net.decay * s["M"] + net.write * torch.bmm((v - got)[:, :, None], s["k"][:, None, :])
        else:
            M = s["M"]
        s = dict(s, v1=mu1, v2=mu2, M=M.detach(), k=k)
    loss = loss / xs.shape[0]
    loss.backward()
    torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
    opt.step()
    return float(loss.detach()), {k: v.detach() for k, v in s.items()}


def ipc_chunk(net, opt, xs, ys, starts, state, steps, rate, beta):
    losses = []
    for t in range(xs.shape[0]):
        state = net.reset(state, starts[t])
        h1, h2, loss0 = settle(net, xs[t], ys[t], state, steps, rate, beta, opt)
        losses.append(loss0)
        with torch.no_grad():
            state = net.advance(state, h1, h2, e=net.E[xs[t]], r=net.recall(state))
    return sum(losses) / len(losses), state


# ------------------------------------------------------------------------------------------------ streams
class Streams:
    """B parallel streams over a list of documents; each document is a token list. A stream that finishes a
    document starts the next one, and marks that step so the cortex forgets the old one."""

    def __init__(self, docs, n, seed=0):
        self.docs, self.n = docs, n
        self.rng = random.Random(seed)
        self.cur = [self._pick() for _ in range(n)]
        self.pos = [0] * n

    def _pick(self):
        return self.docs[self.rng.randrange(len(self.docs))]

    def chunk(self, T):
        xs = torch.zeros(T, self.n, dtype=torch.long)
        ys = torch.zeros(T, self.n, dtype=torch.long)
        st = torch.zeros(T, self.n, dtype=torch.bool)
        for b in range(self.n):
            for t in range(T):
                if self.pos[b] + 1 >= len(self.cur[b]):
                    self.cur[b], self.pos[b] = self._pick(), 0
                    st[t, b] = True
                if self.pos[b] == 0:
                    st[t, b] = True
                xs[t, b] = self.cur[b][self.pos[b]]
                ys[t, b] = self.cur[b][self.pos[b] + 1]
                self.pos[b] += 1
        return xs, ys, st


# ------------------------------------------------------------------------------------------------ measuring
@torch.no_grad()
def perplexity(net, docs, device, batch=256, limit=None):
    """Every next word of every document, each document read from a fresh state."""
    docs = [d for d in docs if len(d) > 1]
    total, count = 0.0, 0
    for s in range(0, len(docs), batch):
        group = docs[s:s + batch]
        L = max(len(d) for d in group)
        x = torch.full((len(group), L), -1, dtype=torch.long)
        for i, d in enumerate(group):
            x[i, :len(d)] = torch.tensor(d)
        x = x.to(device)
        state = net.fresh(len(group), device)
        for t in range(L - 1):
            live = x[:, t + 1] >= 0
            if not bool(live.any()):
                break
            logits, mu1, mu2 = net.forward_step(x[:, t].clamp_min(0), state)
            ce = Fn.cross_entropy(logits, x[:, t + 1].clamp_min(0), reduction="none")
            total += float(ce[live].sum())
            count += int(live.sum())
            state = net.advance(state, mu1, mu2)
        if limit and count >= limit:
            break
    return math.exp(total / max(1, count))


@torch.no_grad()
def score(net, context, candidates, device):
    """Mean log-probability of each candidate reply after the context - how the cortex judges a reply."""
    n = len(candidates)
    state = net.fresh(n, device)
    ctx = torch.tensor(context, device=device)
    for t in range(len(context) - 1):
        _, mu1, mu2 = net.forward_step(ctx[t].expand(n), state)
        state = net.advance(state, mu1, mu2)
    L = max(len(c) for c in candidates)
    seq = torch.full((n, L + 1), -1, dtype=torch.long, device=device)
    seq[:, 0] = ctx[-1]
    for i, c in enumerate(candidates):
        seq[i, 1:len(c) + 1] = torch.tensor(c, device=device)
    tot = torch.zeros(n, device=device)
    cnt = torch.zeros(n, device=device)
    for t in range(L):
        live = seq[:, t + 1] >= 0
        logits, mu1, mu2 = net.forward_step(seq[:, t].clamp_min(0), state)
        lp = -Fn.cross_entropy(logits, seq[:, t + 1].clamp_min(0), reduction="none")
        tot += lp * live
        cnt += live
        state = net.advance(state, mu1, mu2)
    return (tot / cnt.clamp_min(1)).tolist()


def score_lift(net, context, candidates, device, neutral=(SPK_A, SPK_B)):
    """How much the context raises each candidate's probability: log p(reply | context) - log p(reply | nothing).

    Plain mean log-probability rewards whatever is generic ("i'm fine.") after any context at all - run 1 scored
    every model near chance that way, counting included. What makes a reply *the* reply is that this context made
    it more likely than it would otherwise be."""
    with_ctx = score(net, context, candidates, device)
    without = score(net, list(neutral), candidates, device)
    return [a - b for a, b in zip(with_ctx, without)]


@torch.no_grad()
def speak(net, vocab, history_ids, device, drafts=6, max_words=30, temperature=0.8, top_p=0.9, seed=0):
    """Broca: continue the conversation as the next speaker, several drafts, the most fluent one said."""
    g = torch.Generator(device="cpu").manual_seed(seed)
    # the person speaks as <a> and the brain as <b>; a turn ends when the cortex expects the other speaker
    me, other = SPK_B, SPK_A
    ctx = list(history_ids) + [me]
    state = net.fresh(drafts, device)
    for t in range(len(ctx) - 1):
        _, mu1, mu2 = net.forward_step(torch.tensor(ctx[t], device=device).expand(drafts), state)
        state = net.advance(state, mu1, mu2)
    cur = torch.tensor(ctx[-1], device=device).expand(drafts).clone()
    out = [[] for _ in range(drafts)]
    done = torch.zeros(drafts, dtype=torch.bool)
    logp = torch.zeros(drafts)
    for _ in range(max_words):
        logits, mu1, mu2 = net.forward_step(cur, state)
        state = net.advance(state, mu1, mu2)
        pr = torch.softmax(logits.cpu() / temperature, 1)
        pr[:, UNK] = 0
        pr[:, PARA] = 0
        srt, idx = pr.sort(1, descending=True)
        cut = srt.cumsum(1) > top_p
        cut[:, 0] = False
        srt[cut] = 0
        pick = idx.gather(1, torch.multinomial(srt / srt.sum(1, keepdim=True), 1, generator=g)).squeeze(1)
        for i in range(drafts):
            if done[i]:
                continue
            w = int(pick[i])
            if w in (other, me):
                done[i] = True
                continue
            out[i].append(w)
            logp[i] += float(torch.log(pr[i, w] + 1e-12))
        if bool(done.all()):
            break
        cur = pick.to(device)
    # an empty draft said nothing, which is the least fluent thing it could have done
    fluency = [float(logp[i]) / len(out[i]) if out[i] else -float("inf") for i in range(drafts)]
    ranked = sorted(range(drafts), key=lambda i: -fluency[i])
    return [vocab.text(out[i]) for i in ranked], [fluency[i] for i in ranked]


# ------------------------------------------------------------------------------------------------ training
def train(rule, streams, vocab_n, dev_docs, device, steps, T=48, lr=3e-5, seed=0, embed=128, hidden=512,
          memory=True, settle_steps=6, rate=0.2, beta=0.01, eval_every=500, log=print, deadline=None,
          leak=(0.0, 0.0)):
    torch.manual_seed(seed)
    net = OneCortex(vocab_n, embed, hidden, memory=memory, seed=seed, leak=leak).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    state = net.fresh(streams.n, device)
    curve, run, t0 = [], None, time.time()
    for step in range(1, steps + 1):
        xs, ys, st = streams.chunk(T)
        xs, ys, st = xs.to(device), ys.to(device), st.to(device)
        if rule == "bp":
            loss, state = bp_chunk(net, opt, xs, ys, st, state)
        else:
            loss, state = ipc_chunk(net, opt, xs, ys, st, state, settle_steps, rate, beta)
        run = loss if run is None else 0.98 * run + 0.02 * loss
        if step % eval_every == 0 or step == steps:
            ppl = perplexity(net, dev_docs, device, limit=40000)
            curve.append({"step": step, "words": step * T * streams.n, "train_loss": round(run, 4),
                          "dev_ppl": round(ppl, 2), "seconds": round(time.time() - t0)})
            log(f"  [{rule}{'' if memory else '-nomem'} leak {leak} seed {seed} lr {lr}] {curve[-1]}")
            if not math.isfinite(ppl) or ppl > 1e5:
                log("  diverged - stopping")
                break
        if deadline and time.time() > deadline:
            log("  out of time - stopping here")
            break
    return net, curve

In [ ]:
%%writefile /kaggle/working/brain/brain/learn_onecortex.py
"""Train the one cortex on a stream of conversation and reading, and test it against counting and against the
brain it would replace.

    python brain/learn_onecortex.py --wiki <dir with wiki.train.raw ...> --dialogue <dir with train.csv ...> \
        --arms ipc ipc-nomem --seed-offset 0 --out results/onecortex

Pre-registered in PREREGISTRATION_onecortex.md.
"""
import argparse
import collections
import glob
import json
import math
import os
import random
import sys
import time

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, os.path.dirname(HERE))

import numpy as np  # noqa: E402
import torch  # noqa: E402

from brain import onecortex as O  # noqa: E402
from brain.conversation import parse_dailydialog  # noqa: E402
from brain.kneser import KneserNey  # noqa: E402

OLD_BRAIN_R_AT_1 = 0.649      # the self-trained brain's reply selection on the same test items (session 2 end)
LEAK = (0.5, 0.9)             # layer 1 fast, layer 2 slow: higher areas integrate over longer timescales
# arm -> (rule, fast memory on, leak). ipc is run 1's rule (no leak, so each synapse sees only its present input);
# eprop adds leaky neurons and eligibility traces; bp is the yardstick, same architecture as eprop.
ARMS = {"bp": ("bp", True, LEAK), "ipc": ("ipc", True, (0.0, 0.0)), "eprop": ("ipc", True, LEAK),
        "eprop-nomem": ("ipc", False, LEAK)}
GRIDS = {"bp": [3e-4, 1e-3, 3e-3], "ipc": [1e-5, 3e-5, 1e-4, 3e-4], "eprop": [1e-5, 3e-5, 1e-4, 3e-4],
         "eprop-nomem": [1e-5, 3e-5, 1e-4, 3e-4]}


def dialogue_items(dtest, seed=7, limit=2000):
    """The same fixed test items the self-trained brain was examined on: context, ten candidates, which is real."""
    pairs = [(a, b) for t in dtest for a, b in zip(t, t[1:])]
    rng = random.Random(seed)
    replies = [b for _, b in pairs]
    items = []
    for a, b in pairs:
        ds = []
        while len(ds) < 9:
            d = rng.choice(replies)
            if d != b and d not in ds:
                ds.append(d)
        cands = [b] + ds
        rng.shuffle(cands)
        items.append((a, cands, cands.index(b)))
    return items[:limit]


def wiki_docs(path, vocab, limit_words, max_len=1500):
    docs, cur, n = [], [], 0
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line.startswith("= ") and not line.startswith("= = "):
                if len(cur) > 20:
                    docs.append(cur)
                cur = []
                continue
            if not line or line.startswith("="):
                continue
            ids = vocab.ids(line.replace("@-@", "-").replace("@,@", ",").replace("@.@", "."))
            cur += ids + [O.PARA]
            n += len(ids) + 1
            while len(cur) > max_len:
                docs.append(cur[:max_len])
                cur = cur[max_len:]
            if n >= limit_words:
                break
    if len(cur) > 20:
        docs.append(cur)
    return docs


class Mixed:
    """Half the streams hear conversation, half hear reading; one chunk is both, side by side."""

    def __init__(self, talk, read, n, seed):
        self.a = O.Streams(talk, n // 2, seed)
        self.b = O.Streams(read, n - n // 2, seed + 1)
        self.n = n

    def chunk(self, T):
        xa, ya, sa = self.a.chunk(T)
        xb, yb, sb = self.b.chunk(T)
        return torch.cat([xa, xb], 1), torch.cat([ya, yb], 1), torch.cat([sa, sb], 1)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--wiki", required=True)
    ap.add_argument("--dialogue", required=True)
    ap.add_argument("--out", default="results/onecortex")
    ap.add_argument("--arms", nargs="*", default=["bp", "eprop", "ipc"])
    ap.add_argument("--lr", nargs="*", default=[], help="arm=lr, chosen by the grid stage")
    ap.add_argument("--grid", nargs="*", default=[], help="grid stage: arm=lr pairs to try on a short run each")
    ap.add_argument("--grid-steps", type=int, default=400)
    ap.add_argument("--seeds", type=int, default=1)
    ap.add_argument("--seed-offset", type=int, default=0)
    ap.add_argument("--steps", type=int, default=8000)
    ap.add_argument("--streams", type=int, default=128)
    ap.add_argument("--T", type=int, default=48)
    ap.add_argument("--vocab", type=int, default=16000)
    ap.add_argument("--wiki-words", type=int, default=30_000_000)
    ap.add_argument("--hours", type=float, default=10.5)
    ap.add_argument("--lr-ipc", type=float, default=3e-5)
    ap.add_argument("--lr-bp", type=float, default=1e-3)
    ap.add_argument("--items", type=int, default=2000)
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    deadline = time.time() + a.hours * 3600
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    logf = open(os.path.join(a.out, "log.txt"), "a", encoding="utf-8")

    def log(*x):
        line = " ".join(str(v) for v in x)
        print(line, flush=True)
        logf.write(line + "\n")
        logf.flush()

    dtrain = parse_dailydialog(os.path.join(a.dialogue, "train.csv"))
    dval = parse_dailydialog(os.path.join(a.dialogue, "validation.csv"))
    dtest = parse_dailydialog(os.path.join(a.dialogue, "test.csv"))
    counts = collections.Counter()
    for t in dtrain:
        for turn in t:
            for w in O.words(turn):
                counts[w] += 20                   # conversation's words must be in the vocabulary
    with open(os.path.join(a.wiki, "wiki.train.raw"), encoding="utf-8") as f:
        for i, line in enumerate(f):
            counts.update(O.words(line))
            if i > 400_000:
                break
    voc = O.Vocab(counts, a.vocab)
    talk = [voc.dialogue(t) for t in dtrain]
    read = wiki_docs(os.path.join(a.wiki, "wiki.train.raw"), voc, a.wiki_words)
    dev_docs = [voc.dialogue(t) for t in dval]
    test_talk = [voc.dialogue(t) for t in dtest]
    test_read = wiki_docs(os.path.join(a.wiki, "wiki.test.raw"), voc, 10 ** 9)
    log(f"device {dev}; vocabulary {len(voc):,}; conversations {len(talk):,} "
        f"({sum(map(len, talk)):,} words); reading {len(read):,} docs ({sum(map(len, read)):,} words)")
    items = dialogue_items(dtest, limit=a.items)
    budget = a.steps * a.T * a.streams

    # counting: Kneser-Ney 5-gram on as many words as the cortex hears, in the same mix
    t0 = time.time()
    rng = random.Random(0)
    half = budget // 2
    talk_stream, read_stream = [], []
    while len(talk_stream) < half:
        talk_stream += rng.choice(talk)
    for d in read:
        if len(read_stream) >= half:
            break
        read_stream += d
    kn = KneserNey(5).fit(np.array(talk_stream[:half] + read_stream[:half]), len(voc))

    def kn_ppl(docs):
        lp, n = 0.0, 0
        for d in docs:
            if len(d) > 1:
                x = kn.logprob(np.array(d), 1)
                lp += float(x.sum())
                n += len(x)
        return math.exp(-lp / max(1, n))

    def kn_select():
        """Reply selection by how much the context raises the reply's probability, as for the cortex."""
        hits = 0
        neutral = [O.SPK_A, O.SPK_B]
        for ctx, cands, gold in items:
            c = [O.SPK_A] + voc.ids(ctx) + [O.SPK_B]
            sc = []
            for cand in cands:
                ids = voc.ids(cand) or [O.UNK]
                sc.append(float(kn.logprob(np.array(c + ids), len(c)).mean())
                          - float(kn.logprob(np.array(neutral + ids), len(neutral)).mean()))
            hits += int(np.argmax(sc)) == gold
        return hits / len(items)

    baseline = {"kn_talk_ppl": kn_ppl(test_talk), "kn_read_ppl": kn_ppl(test_read[:200]),
                "kn_r_at_1": kn_select() if not a.grid else None, "old_brain_r_at_1": OLD_BRAIN_R_AT_1}
    log(f"Kneser-Ney 5-gram on {len(talk_stream[:half]) + len(read_stream[:half]):,} words: {baseline} "
        f"({time.time() - t0:.0f}s)")
    del kn
    results = {"config": vars(a), "vocab": len(voc), "baseline": baseline, "runs": []}

    def save():
        json.dump(results, open(os.path.join(a.out, "results.json"), "w"), indent=1)

    save()
    if a.grid:
        # the grid stage: each arm and learning rate on a short run, validation perplexity only
        results["grid"] = []
        for pair in a.grid:
            arm, lr = pair.split("=")
            rule, memory, leak = ARMS[arm]
            net, curve = O.train(rule, Mixed(talk, read, a.streams, 0), len(voc), dev_docs, dev, a.grid_steps, T=a.T,
                                 lr=float(lr), seed=0, memory=memory, leak=leak, eval_every=a.grid_steps, log=log)
            results["grid"].append({"arm": arm, "lr": float(lr), "dev_ppl": curve[-1]["dev_ppl"] if curve else None})
            save()
            del net
        return
    chosen = {k: float(v) for k, v in (x.split("=") for x in a.lr)}
    for seed in range(a.seed_offset, a.seed_offset + a.seeds):
        for arm in a.arms:
            if deadline - time.time() < 1200:
                log("stopping: under 20 minutes left")
                break
            rule, memory, leak = ARMS[arm]
            streams = Mixed(talk, read, a.streams, seed)
            t0 = time.time()
            lr = chosen.get(arm, a.lr_bp if rule == "bp" else a.lr_ipc)
            net, curve = O.train(rule, streams, len(voc), dev_docs, dev, a.steps, T=a.T, lr=lr, seed=seed,
                                 memory=memory, leak=leak, eval_every=max(250, a.steps // 12), log=log,
                                 deadline=deadline - 900)
            net.eval()
            talk_ppl = O.perplexity(net, test_talk, dev)
            read_ppl = O.perplexity(net, test_read[:200], dev)
            hits = 0
            for ctx, cands, gold in items:
                c = [O.SPK_A] + voc.ids(ctx) + [O.SPK_B]
                sc = O.score_lift(net, c, [voc.ids(x) or [O.UNK] for x in cands], dev)
                hits += int(np.argmax(sc)) == gold
            samples = {}
            for prompt in ["hello , how are you ?", "tell me a joke .", "i lost my job yesterday .",
                           "what do you want to eat tonight ?", "i think my cat is plotting against me .",
                           "do you like pizza ?"]:
                said, _ = O.speak(net, voc, [O.SPK_A] + voc.ids(prompt), dev, seed=seed)
                samples[prompt] = said[:3]
            run = {"arm": arm, "seed": seed, "lr": lr, "leak": leak, "curve": curve, "talk_ppl": round(talk_ppl, 2),
                   "read_ppl": round(read_ppl, 2), "r_at_1": round(hits / len(items), 4),
                   "seconds": round(time.time() - t0), "params": net.params(), "samples": samples,
                   "finished": bool(curve) and curve[-1]["step"] == a.steps}
            results["runs"].append(run)
            log(f"== {arm} seed {seed}: talk ppl {run['talk_ppl']} read ppl {run['read_ppl']} R@1 {run['r_at_1']} "
                f"({run['seconds']}s) | hello -> {samples['hello , how are you ?'][:1]}")
            torch.save({"state": {k: v.cpu() for k, v in net.state_dict().items()}, "itos": voc.itos,
                        "config": {"embed": net.d, "hidden": net.H, "mem": net.m, "memory": net.memory,
                                   "leak": [net.a1, net.a2]},
                        "arm": arm, "seed": seed, "talk_ppl": talk_ppl},
                       os.path.join(a.out, f"onecortex_{arm}_seed{seed}.pt"))
            save()
            del net
        else:
            continue
        break
    save()


def analyse(runs, baseline):
    """The pre-registered verdicts (PREREGISTRATION_onecortex.md, amendment 1). Two seeds, so each hypothesis is
    decided by every finished seed clearing its bar."""
    by = collections.defaultdict(dict)
    for r in runs:
        if r.get("finished"):
            by[r["arm"]][r["seed"]] = r
    out = {}
    ep, ipc = by.get("eprop", {}), by.get("ipc", {})
    both = sorted(set(ep) & set(ipc))
    if both:
        out["H1 eligibility traces beat run 1's rule on conversation perplexity"] = {
            "values": [(ep[s]["talk_ppl"], ipc[s]["talk_ppl"]) for s in both],
            "verdict": "SUPPORTED" if all(ep[s]["talk_ppl"] < ipc[s]["talk_ppl"] for s in both) else "not supported"}
    if ep:
        out["H2 eprop beats Kneser-Ney on conversation perplexity"] = {
            "values": [ep[s]["talk_ppl"] for s in sorted(ep)], "bar": round(baseline["kn_talk_ppl"], 2),
            "verdict": "SUPPORTED" if all(ep[s]["talk_ppl"] < baseline["kn_talk_ppl"] for s in ep) else
            "not supported"}
        if baseline.get("kn_r_at_1") is not None:
            out["H3 eprop picks the real reply more often than Kneser-Ney (context lift)"] = {
                "values": [ep[s]["r_at_1"] for s in sorted(ep)], "bar": baseline["kn_r_at_1"],
                "verdict": "SUPPORTED" if all(ep[s]["r_at_1"] > baseline["kn_r_at_1"] for s in ep) else
                "not supported"}
    return out


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/brain/brain/kneser.py
"""The counting baseline: interpolated Kneser-Ney (Kneser & Ney 1995; Chen & Goodman 1999).

Any claim that a learned cortex predicts the next word well has to beat counting, because counting is what the
brain's earlier self-trained parts did and what an n-gram model has done since the 1990s. KN is the strongest
classical form of it: absolute discounting, and lower orders estimated from how many *different* contexts a word
follows ("Francisco" is common but almost only after "San", so it should be a poor guess after anything else).

Vectorised: every n-gram is folded into one 64-bit key (a random polynomial hash; collisions are negligible at
these sizes), counted with numpy.unique, and looked up with searchsorted, so tens of millions of tokens fit in a
few minutes and a few GB.
"""
import math

import numpy as np

_M = np.random.default_rng(12345).integers(1, 2 ** 63 - 1, size=16, dtype=np.int64).astype(np.uint64) | np.uint64(1)


def _keys(tokens, k, end):
    """Key of the k tokens ending at each position in `end` (exclusive)."""
    key = np.zeros(len(end), np.uint64)
    t = tokens.astype(np.uint64)
    for j in range(k):
        key = key * np.uint64(1000003) + (t[end - k + j] + np.uint64(1)) * _M[j]
    return key + np.uint64(k)


class Table:
    def __init__(self, keys, values):
        order = np.argsort(keys)
        self.k, self.v = keys[order], values[order]

    def get(self, q):
        i = np.searchsorted(self.k, q)
        i = np.minimum(i, len(self.k) - 1)
        hit = self.k[i] == q
        out = np.zeros(len(q), self.v.dtype if self.v.ndim == 1 else self.v.dtype)
        if self.v.ndim == 1:
            out = np.where(hit, self.v[i], 0)
        else:
            out = np.where(hit[:, None], self.v[i], 0)
        return out


class KneserNey:
    def __init__(self, order=5):
        self.n = order

    def fit(self, tokens, vocab):
        tokens = np.asarray(tokens, np.int64)
        self.V = vocab
        T = len(tokens)
        self.ngram, self.context, self.D = {}, {}, {}
        for k in range(1, self.n + 1):
            end = np.arange(k, T + 1)
            g = _keys(tokens, k, end)
            if k == self.n:
                # the highest order uses raw counts
                uniq, first, counts = np.unique(g, return_index=True, return_counts=True)
                ends = end[first]
            else:
                # lower orders use continuation counts: how many distinct words came before this k-gram
                end1 = np.arange(k + 1, T + 1)
                g1 = _keys(tokens, k + 1, end1)
                u1, f1 = np.unique(g1, return_index=True)
                suffix = _keys(tokens, k, end1[f1])
                uniq, first, counts = np.unique(suffix, return_index=True, return_counts=True)
                ends = end1[f1][first]
            n1, n2 = int((counts == 1).sum()), int((counts == 2).sum())
            self.D[k] = n1 / (n1 + 2 * n2) if n1 + 2 * n2 else 0.5
            self.ngram[k] = Table(uniq, counts.astype(np.float64))
            if k == 1:
                self.unigram_total = float(counts.sum())
                self.unigram_types = float(len(counts))
            else:
                ctx = _keys(tokens, k - 1, ends - 1)
                cu, inv = np.unique(ctx, return_inverse=True)
                total = np.bincount(inv, weights=counts.astype(np.float64))
                types = np.bincount(inv).astype(np.float64)
                self.context[k] = Table(cu, np.stack([total, types], 1))
        return self

    def logprob(self, tokens, start):
        """log P(tokens[i] | the n-1 before it) for every i >= start."""
        tokens = np.asarray(tokens, np.int64)
        pos = np.arange(start, len(tokens))
        D1 = self.D[1]
        c1 = self.ngram[1].get(_keys(tokens, 1, pos + 1))
        p = (np.maximum(c1 - D1, 0) + D1 * self.unigram_types / self.V) / self.unigram_total
        for k in range(2, self.n + 1):
            ok = pos - (k - 1) >= 0
            c = self.ngram[k].get(_keys(tokens, k, pos + 1))
            ctx = self.context[k].get(_keys(tokens, k - 1, pos))
            total, types = ctx[:, 0], ctx[:, 1]
            seen = (total > 0) & ok
            D = self.D[k]
            higher = np.where(seen, (np.maximum(c - D, 0) + D * types * p) / np.where(seen, total, 1), p)
            p = higher
        return np.log(np.maximum(p, 1e-12))

    def perplexity(self, tokens, start=8):
        return math.exp(-float(self.logprob(tokens, start).mean()))

In [ ]:
%%writefile /kaggle/working/brain/brain/conversation.py
"""Talking: what people said after something like this, and which of those fits now.

A person's first reply to "how is your day?" is not composed from a grammar; it is recalled - they have heard
that exchange a thousand times. So the conversational memory stores exchanges as episodes (what was said, what
was said back), recalls the ones whose opening resembles what was just said, and the striatum chooses among the
replies they offer. Nothing about how to reply is written here: which reply fits is learned by practice, with
the reply that was actually given as the outcome.

Three things describe how well a reply fits, all learned from what was heard:

  recall      how similar the moment is to moments this reply (or one like it) followed before - hippocampal
              episodic recall, the kNN-LM idea (Khandelwal et al. 2020) applied to turns instead of words.
  association the Hebbian link between words across a turn boundary: "thanks" tends to be answered by
              "welcome", "how are you" by "fine". Counted, and scored as pointwise mutual information.
  overlap     plain shared words - the baseline every retrieval dialogue system has to beat (TF-IDF, Lowe et
              al. 2015).

The striatum learns how to weigh them, and a few thousand lexical pairs across the boundary besides, from
practice: shown a moment and ten candidate replies, one of which was really said.
"""
import collections
import math
import re

import numpy as np
import scipy.sparse as sp

WORD = re.compile(r"[a-z0-9']+|[?!.]")
QUOTED = re.compile(r"'([^']*)'|\"([^\"]*)\"")


def words(text):
    return WORD.findall(text.lower())


def parse_dailydialog(path):
    """DailyDialog as Kaggle ships it: one dialogue per row, turns as a numpy-style list of strings."""
    import csv
    csv.field_size_limit(1 << 30)
    out = []
    with open(path, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            turns = [" ".join((a or b).split()) for a, b in QUOTED.findall(row["dialog"])]
            turns = [t for t in turns if t]
            if len(turns) >= 2:
                out.append(turns)
    return out


class Vectoriser:
    """Words -> sparse, rarity-weighted vectors. The rarity is counted from what was heard."""

    def __init__(self):
        self.vocab = {}
        self.df = None
        self.n = 0

    def fit(self, texts):
        df = collections.Counter()
        for t in texts:
            df.update(set(words(t)))
        self.vocab = {w: i for i, (w, c) in enumerate(df.most_common()) if c >= 2}
        self.df = np.zeros(len(self.vocab))
        for w, i in self.vocab.items():
            self.df[i] = df[w]
        self.n = len(texts)
        self.idf = np.log((self.n + 1) / (self.df + 1)) + 1.0
        return self

    def __call__(self, texts, weighted=True, binary=False):
        rows, cols, vals = [], [], []
        for r, t in enumerate(texts):
            c = collections.Counter(w for w in words(t) if w in self.vocab)
            for w, k in c.items():
                i = self.vocab[w]
                rows.append(r)
                cols.append(i)
                v = 1.0 if binary else (1 + math.log(k))
                vals.append(v * (self.idf[i] if weighted else 1.0))
        m = sp.csr_matrix((vals, (rows, cols)), shape=(len(texts), len(self.vocab)))
        if weighted:
            norm = np.sqrt(np.asarray(m.multiply(m).sum(1)).ravel())
            norm[norm == 0] = 1.0
            m = sp.diags(1.0 / norm) @ m
        return m.tocsr()


class Conversation:
    def __init__(self):
        self.contexts, self.replies = [], []
        self.vec = None

    # ------------------------------------------------------------------ hearing
    def hear(self, dialogues):
        """Store every exchange in these dialogues as an episode, and count what answered what."""
        for turns in dialogues:
            for a, b in zip(turns, turns[1:]):
                self.contexts.append(a)
                self.replies.append(b)
        self.vec = Vectoriser().fit(self.contexts + self.replies)
        self.C = self.vec(self.contexts)
        self.R = self.vec(self.replies)
        # association across the turn boundary: co-occurrence of a word before it and a word after it
        Xb = self.vec(self.contexts, weighted=False, binary=True)
        Yb = self.vec(self.replies, weighted=False, binary=True)
        joint = (Xb.T @ Yb).tocoo()
        n = len(self.contexts)
        px = np.asarray(Xb.sum(0)).ravel() / n
        py = np.asarray(Yb.sum(0)).ravel() / n
        keep = joint.data >= 3
        r, c, v = joint.row[keep], joint.col[keep], joint.data[keep] / n
        pmi = np.log(v / (px[r] * py[c]))
        pmi = np.maximum(pmi, 0.0)
        self.PMI = sp.csr_matrix((pmi, (r, c)), shape=(len(self.vec.vocab),) * 2)
        return len(self.contexts)

    # --------------------------------------------------------------- describing
    def recall(self, context, k=30):
        """The stored moments most like this one: (similarity, index)."""
        q = self.vec([context])
        sims = (self.C @ q.T).toarray().ravel()
        top = np.argpartition(-sims, min(k, len(sims) - 1))[:k]
        top = top[np.argsort(-sims[top])]
        return [(float(sims[i]), int(i)) for i in top if sims[i] > 0]

    def describe(self, context, candidates, recalled=None):
        """Features of each candidate reply to this context."""
        q = self.vec([context])
        R = self.vec(candidates)
        overlap = (R @ q.T).toarray().ravel()
        qb = self.vec([context], weighted=False, binary=True)
        Rb = self.vec(candidates, weighted=False, binary=True)
        assoc_raw = np.asarray((Rb @ (self.PMI.T @ qb.T)).todense()).ravel()
        nq = max(1, qb.nnz)
        nr = np.maximum(1, np.diff(Rb.indptr))
        assoc = assoc_raw / np.sqrt(nq * nr)
        recalled = recalled if recalled is not None else self.recall(context)
        if recalled:
            sims = np.array([s for s, _ in recalled])
            past = self.R[[i for _, i in recalled]]
            match = (R @ past.T).toarray()                      # candidate x recalled reply
            knn = (match * sims[None, :]).max(1)
            knn_sum = (match * sims[None, :]).sum(1) / (sims.sum() or 1.0)
        else:
            knn = knn_sum = np.zeros(len(candidates))
        cw = words(context)
        asks = "?" in cw
        c_first = cw[0] if cw else "<none>"
        c_last = [w for w in cw if w not in "?!."][-3:]
        out = []
        for k, cand in enumerate(candidates):
            rw = words(cand)
            r_first = rw[0] if rw else "<none>"
            r_open = rw[:3]
            L = len(rw)
            f = [("overlap", overlap[k]), ("assoc", assoc[k]), ("knn", knn[k]), ("knn_sum", knn_sum[k]),
                 (f"assoc|{asks}", assoc[k]), (f"knn|{asks}", knn[k]),
                 f"b|{asks}", f"rfirst|{asks}|{r_first}", f"pair|{c_first}|{r_first}",
                 f"rlen|{asks}|{min(L // 4, 6)}", f"rq|{asks}|{'?' in rw}",
                 f"knnb|{int(min(knn[k], 0.99) * 10)}", f"assocb|{int(min(assoc[k], 1.99) * 5)}"]
            f += [f"x|{x}|{y}" for x in c_last for y in r_open]
            out.append(f)
        return out

    # ------------------------------------------------------------------ acting
    def practise(self, striatum, context, reply, distractors, rng=None):
        cands = [reply] + list(distractors)
        order = list(range(len(cands)))
        (rng or np.random.default_rng()).shuffle(order)
        cands = [cands[i] for i in order]
        gold = order.index(0)
        enc = striatum.encode(self.describe(context, cands, self._recall_excluding(context, reply)))
        target = np.zeros(len(cands))
        target[gold] = 1.0
        s = striatum.scores(enc, consolidated=False)
        top = np.flatnonzero(s == s.max())
        hit = int(top[0]) == gold and len(top) == 1
        striatum.teach(enc, target)
        return hit

    def _recall_excluding(self, context, reply, k=30):
        """Recall without the episode being practised on - remembering the answer is not knowing it."""
        got = self.recall(context, k + 2)
        return [(s, i) for s, i in got if not (self.replies[i] == reply and self.contexts[i] == context)][:k]

    def choose(self, striatum, context, candidates, recalled=None):
        enc = striatum.encode(self.describe(context, candidates, recalled))
        k, s = striatum.choose(enc)
        return k, s

    def reply(self, striatum, context, pool=40):
        """Say something: the replies that followed moments like this one, and the striatum's pick of them."""
        recalled = self.recall(context, pool)
        if not recalled:
            return None, 0.0
        seen, cands = set(), []
        for _, i in recalled:
            r = self.replies[i]
            if r not in seen:
                seen.add(r)
                cands.append(r)
        k, s = self.choose(striatum, context, cands, recalled)
        return cands[k], float(s[k])

In [ ]:
%%writefile /kaggle/working/brain/tests/check_onecortex.py
"""The one cortex, checked where it can be checked exactly (runs on Kaggle, before any training)."""
import os
import sys

import torch
import torch.nn.functional as Fn

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, os.path.dirname(HERE))

from brain import onecortex as O  # noqa: E402

FAILED = []


def check(name, cond, detail=""):
    print(f"{'PASS' if cond else 'FAIL'}  {name}" + (f"   {detail}" if detail and not cond else ""))
    if not cond:
        FAILED.append(name)


def cos(a, b):
    return float(Fn.cosine_similarity(a.flatten(), b.flatten(), dim=0))


torch.manual_seed(0)
net = O.OneCortex(40, embed=8, hidden=16, mem=8, memory=True, seed=1)
n = 32
x = torch.randint(0, 40, (n,))
y = torch.randint(0, 40, (n,))
state = net.fresh(n, "cpu")
state["v1"] = torch.randn(n, 16)
state["v2"] = torch.randn(n, 16)
state["M"] = 0.1 * torch.randn(n, 8, 8)
state["k"] = Fn.normalize(torch.randn(n, 8), dim=1)

# 1. with the previous state as given, settled local changes point where the one-step gradient points
logits, _, _ = net.forward_step(x, state)
Fn.cross_entropy(logits, y).backward()
true = {k: getattr(net, k).grad.clone() for k in ("Wo", "W2", "U2", "Wm", "W1", "U1", "E")}
net.zero_grad()
h1, h2, _ = O.settle(net, x, y, state, steps=40, rate=0.2, beta=0.01)
cur = O.local_changes(net, x, y, state, h1, h2, 0.01)
g = O.local_grads(net, x, state, h1, h2, cur, 0.01)
for k in true:
    c = cos(g[k], true[k])
    check(f"local change of {k} points where the gradient points", c > 0.9, f"{c:.3f}")

# 2. the hippocampus: writing the same moment again and again makes recall converge on it
st = net.fresh(1, "cpu")
st["k"] = Fn.normalize(torch.randn(1, 8), dim=1)
h2 = torch.randn(1, 16)
want = Fn.linear(torch.tanh(h2), net.Pv)
key = st["k"].clone()
for _ in range(20):
    nxt = net.advance(st, torch.randn(1, 16), h2)
    st = dict(nxt, k=key)
got = net.recall(st)
check("fast memory recalls what followed a key", cos(got, want) > 0.95, f"{cos(got, want):.3f}")
off = O.OneCortex(40, embed=8, hidden=16, mem=8, memory=False, seed=1)
check("and a cortex without it recalls nothing", float(off.recall(st).abs().sum()) == 0.0)

# 3. a new conversation starts clean, and only for the streams that started one
st = net.fresh(3, "cpu")
st["M"] += 1.0
st["v1"] += 1.0
st["zU1"] += 1.0
st = net.reset(st, torch.tensor([False, True, False]))
check("reset empties the memory of the stream that started over",
      float(st["M"][1].abs().sum()) == 0 and float(st["M"][0].abs().sum()) > 0)
check("and quiets its cortex and its traces", float(st["v1"][1].abs().sum()) == 0
      and float(st["zU1"][1].abs().sum()) == 0 and float(st["v1"][2].abs().sum()) > 0)

# 4. streams mark where a document starts
streams = O.Streams([[1, 5, 6, 7], [2, 8, 9, 10, 11]], 2, seed=0)
xs, ys, st_ = streams.chunk(12)
check("streams produce next-word targets", bool((ys[:-1] == xs[1:])[~st_[1:]].all()))
check("and flag document starts", bool(st_[0].all()) and int(st_.sum()) > 2)

# 5. it learns a repeating conversation with the local rule alone
docs = [[O.SPK_A, 10, 11, 12, O.SPK_B, 13, 14, 15, 16] * 4]
before = O.perplexity(O.OneCortex(40, embed=8, hidden=16, mem=8, seed=2), docs, "cpu")
learned, curve = O.train("ipc", O.Streams(docs, 8, seed=0), 40, docs, "cpu", steps=60, T=12, lr=3e-3,
                         seed=2, embed=8, hidden=16, settle_steps=4, eval_every=60, log=lambda *a: None)
after = curve[-1]["dev_ppl"]
check("incremental predictive coding learns a conversation it keeps hearing", after < before * 0.5,
      f"{before:.1f} -> {after:.1f}")

# 6. it can be asked to judge and to speak
sc = O.score(learned, [O.SPK_A, 10, 11, 12, O.SPK_B], [[13, 14, 15], [30, 31, 32]], "cpu")
check("it prefers the reply it has heard before", sc[0] > sc[1], str(sc))


class Tiny:
    itos = O.SPECIAL + [f"w{i}" for i in range(36)]


said, fluency = O.speak(learned, type("V", (), {"text": lambda self, ids: " ".join(Tiny.itos[i] for i in ids)})(),
                        [O.SPK_A, 10, 11, 12], "cpu", drafts=3, max_words=8)
check("it speaks: several drafts, most fluent first", len(said) == 3 and fluency[0] >= fluency[-1], str(said))

# 7. eligibility traces: a synapse's memory of its input fades at its neuron's leak, and with no leak the rule is
# exactly run 1's (the trace is just the present input)
leaky = O.OneCortex(40, embed=8, hidden=16, mem=8, seed=3, leak=(0.5, 0.9))
st = leaky.fresh(1, "cpu")
h = torch.zeros(1, 16)
st = leaky.advance(st, h, h, e=torch.ones(1, 8), r=torch.zeros(1, 8))
first = float(st["zW1"].mean())
st = leaky.advance(st, h, h, e=torch.zeros(1, 8), r=torch.zeros(1, 8))
check("a trace fades at the neuron's own leak", abs(float(st["zW1"].mean()) - 0.5 * first) < 1e-6,
      f"{first} -> {float(st['zW1'].mean())}")
z = net.traces_now(torch.ones(4, 8), net.fresh(4, "cpu"), torch.zeros(4, 16), torch.zeros(4, 8))
check("with no leak the trace is the present input", torch.allclose(z["zW1"], torch.ones(4, 8)))

# 8. with leaky neurons and traces it learns a dependency across a gap of filler words
docs = [[O.SPK_A, c, 20, 21, 22, c + 10] for c in range(5, 10)] * 6
before = O.perplexity(O.OneCortex(40, embed=8, hidden=16, mem=8, seed=4, leak=(0.5, 0.9)), docs, "cpu")
_, curve = O.train("ipc", O.Streams(docs, 8, seed=1), 40, docs, "cpu", steps=80, T=12, lr=3e-3, seed=4, embed=8,
                   hidden=16, settle_steps=4, eval_every=80, log=lambda *a: None, leak=(0.5, 0.9))
check("leaky cortex with traces learns across a gap", curve[-1]["dev_ppl"] < before * 0.6,
      f"{before:.1f} -> {curve[-1]['dev_ppl']:.1f}")
lift = O.score_lift(learned, [O.SPK_A, 10, 11, 12, O.SPK_B], [[13, 14, 15], [30, 31, 32]], "cpu")
check("context lift prefers the reply this context made likely", lift[0] > lift[1], str(lift))

print("\n" + ("ALL PASS" if not FAILED else f"{len(FAILED)} FAILED: " + ", ".join(FAILED)))
sys.exit(1 if FAILED else 0)

In [ ]:
%%writefile /kaggle/working/brain/PREREGISTRATION_onecortex.md
# Pre-registration: the one cortex

Written 2026-09-27, before any run of `brain/onecortex.py` (no local runs; the Kaggle notebook runs the checks
first). Code at the commit this file is in.

## Why

On 20 off-the-wall lines the self-trained brain gave about 2 acceptable replies. Its failures are architectural:
it selects rather than produces, matches words rather than meaning, has no conversational state, and sits behind
hand-written routing. Its only learned parts are one-layer choosers over hand-written features. The one cortex
replaces that with a single recurrent network that hears the conversation as a stream, predicts it, remembers it
in a fast hippocampal memory, and speaks from the same tissue - learning by incremental predictive coding (a local
rule; run 3 of the deep cortex showed it improving steadily where ordinary predictive coding drifted).

## Design

- **Network.** Embedding 128, two recurrent tanh layers of 512 (each layer predicted from the layer below and from
  its own previous activity), a 64-dimensional delta-rule fast memory read by the top layer (keys and values
  through fixed random projections; decay 0.98; wiped at the start of each conversation), softmax over 16,000
  lowercase words plus speaker marks `<a>`/`<b>` and paragraph `<p>`.
- **Stream.** 128 parallel streams, half DailyDialog training conversations (speaker-marked), half WikiText-103
  articles (up to 30M words); chunks of 48 words; state carried across chunks and reset when a new document starts.
- **Rules.** `ipc`: 6 energy-descending settling steps per word, synapses change at every settling step from local
  errors (beta 0.01), Adam lr 3e-5 (run 3's grid choice for ipc), no backprop and none through time. `ipc-nomem`:
  the same with the fast memory switched off. `bp`: truncated backprop through each chunk, Adam lr 1e-3 - a
  yardstick only, never part of the brain.
- **Budget.** 8,000 chunks (49M words heard), or less if the session runs short; the log says which.
- **Baselines.** Kneser-Ney 5-gram fit on as many words as the cortex hears, in the same half-conversation,
  half-reading mix. The old self-trained brain's reply selection on the same test items: R@1 0.649.
- **Measures.** Perplexity on every next word of every DailyDialog *test* conversation, each from a fresh state
  (primary); WikiText-103 test perplexity (first 200 articles; secondary); reply selection on the first 2,000 of
  the self-trained brain's fixed DailyDialog test items (the real reply among ten, scored by mean log-probability
  of the reply after the context); six sampled replies to fixed prompts.
- **Seeds.** Two (0 and 1), one per GPU: seed 0 runs bp, ipc, ipc-nomem; seed 1 runs ipc, ipc-nomem.

## Hypotheses (decided by every finished seed clearing the bar; with two seeds a t-test is powerless)

- **H1.** ipc beats Kneser-Ney on conversation (DailyDialog test) perplexity.
- **H2.** ipc picks the real reply more often than the old brain (R@1 > 0.649).
- **H3.** The hippocampal fast memory helps: ipc below ipc-nomem on conversation perplexity.

## How it will be read

H1 is the bar for the cortex to be worth speaking from at all. H2 is a high bar - the old brain was built for
exactly that task - and failing it with H1 holding would mean the cortex models conversation better than
counting but does not yet judge replies as well as a purpose-built chooser. Samples are reported as they come
out, not selected. Whatever the result, the next step is written down before the run is read: if H1 holds the
cortex goes into the portal as Broca and is trained longer; if not, the report says how far behind it is and
whether its curve is still falling.

## Amendment 1, 2026-09-28: run 1's result, and the test of eligibility traces

**Run 1 (reported in full).** DailyDialog test perplexity: Kneser-Ney 32.5; bp (same architecture, yardstick)
35.4, with sensible replies of its own ("i'm fine. thank you.", "really? how do you feel about it?"); ipc 236.0 /
188.2; ipc without fast memory 182.1 / 215.5. H1, H2, H3 not supported. ipc sat near 200 from the first checkpoint
and never improved: with credit reaching only one word back, the recurrent synapses cannot learn to use the past.
Two mistakes of mine, corrected below: (1) ipc's learning rate was carried over from the 8-word network untuned;
(2) reply selection by mean log-probability favours generic replies - every model, counting included, scored near
chance (0.12-0.18), so H2 measured the scoring rule, not the models.

**Run 2 (this test).** The one cortex gets leaky neurons (membrane potential carries 0.5 of itself into the next
word in layer 1 and 0.9 in layer 2 - higher areas integrate longer) and **eligibility traces**: every synapse keeps a
fading memory of its own input at its neuron's leak, and changes by its layer's prediction error times that trace
(e-prop, Bellec et al. 2020). With no leak this is exactly run 1's rule, which is kept as the `ipc` arm.

- Arms: `eprop` (the proposal), `ipc` (run 1's rule), `bp` (yardstick; same leaky architecture, truncated BPTT).
- Learning rates: a grid per arm on 400-step runs (bp 3e-4/1e-3/3e-3; ipc and eprop 1e-5/3e-5/1e-4/3e-4), chosen on
  DailyDialog validation perplexity, the same way for every arm, before the main runs.
- Main runs: 5,000 chunks each (30.7M words), seeds 0 and 1 (seed 0: bp, eprop, ipc; seed 1: eprop, ipc).
- Reply selection is now scored by **context lift** - log p(reply | context) minus log p(reply | no context) - for
  the cortex and for Kneser-Ney alike, on the same 2,000 fixed test items.

Hypotheses (every finished seed must clear the bar):

- **H1.** Eligibility traces beat run 1's rule: eprop below ipc on conversation perplexity, seed for seed.
- **H2.** eprop beats Kneser-Ney on conversation perplexity.
- **H3.** eprop picks the real reply more often than Kneser-Ney does, both scored by context lift.

Reported regardless: bp, the whole curves, the grid, and unselected samples. The old brain's 0.649 is quoted but is
no longer a hypothesis: it was measured with a different scorer.

### Checks

In [ ]:
r = subprocess.run([sys.executable, "tests/check_onecortex.py"], capture_output=True, text=True, cwd=W)
print(r.stdout[-3000:], r.stderr[-3000:])
assert r.returncode == 0, "checks failed - not training a broken cortex"

### The run

In [ ]:
import torch
START = time.time()
gpus = torch.cuda.device_count()
G1 = "1" if gpus > 1 else "0"


def launch(jobs):
    procs = []
    for dev, args, out in jobs:
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=dev)
        cmd = [sys.executable, "-u", "brain/learn_onecortex.py", "--wiki", WIKI, "--dialogue", TALK, "--out", out,
               "--steps", "5000", *args]
        procs.append(subprocess.Popen(cmd, cwd=W, env=env, stdout=open(out + ".log", "w"), stderr=subprocess.STDOUT))
    t0 = time.time()
    while any(p.poll() is None for p in procs):
        time.sleep(300)
        for *_, out in jobs:
            lines = open(out + ".log").read().splitlines()
            print(f"[{(time.time() - t0) / 3600:.1f}h] {out[-7:]}:", *[l for l in lines if l.startswith(("==", "  [", "Kneser", "device"))][-2:], sep="\n  ", flush=True)
    for *_, out in jobs:
        print(open(out + ".log").read()[-2500:])


# stage 1: the same learning-rate grid for every rule, on short runs, chosen on validation
launch([("0", ["--grid", "bp=3e-4", "bp=1e-3", "bp=3e-3", "eprop=1e-5", "eprop=3e-5", "ipc=1e-5", "ipc=3e-5",
               "--grid-steps", "400", "--hours", "3"], "/kaggle/working/grid_a"),
        (G1, ["--grid", "eprop=1e-4", "eprop=3e-4", "ipc=1e-4", "ipc=3e-4", "--grid-steps", "400",
              "--hours", "3"], "/kaggle/working/grid_b")])
grid = json.load(open("/kaggle/working/grid_a/results.json")).get("grid", []) + \
       json.load(open("/kaggle/working/grid_b/results.json")).get("grid", [])
best = {}
for g in grid:
    if g["dev_ppl"] is not None and (g["arm"] not in best or g["dev_ppl"] < best[g["arm"]][1]):
        best[g["arm"]] = (g["lr"], g["dev_ppl"])
print("grid:", json.dumps(grid, indent=1))
print("chosen:", best)
LRS = [f"{k}={v[0]}" for k, v in best.items()]

# stage 2: the pre-registered runs with the chosen rates, two seeds, one per GPU
left = 11.3 - (time.time() - START) / 3600
launch([("0", ["--arms", "bp", "eprop", "ipc", "--seed-offset", "0", "--lr", *LRS, "--hours", f"{left:.2f}"],
         "/kaggle/working/one_a"),
        (G1, ["--arms", "eprop", "ipc", "--seed-offset", "1", "--lr", *LRS, "--hours", f"{left:.2f}"],
         "/kaggle/working/one_b")])

### Verdicts and what it says

In [ ]:
sys.path.insert(0, W)
from brain.learn_onecortex import analyse
A = json.load(open("/kaggle/working/one_a/results.json"))
B = json.load(open("/kaggle/working/one_b/results.json"))
runs = A["runs"] + B["runs"]
v = analyse(runs, A["baseline"])
print("grid choices:", {r["arm"]: r.get("lr") for r in runs})
json.dump({"baseline": A["baseline"], "runs": runs, "verdict": v}, open("/kaggle/working/onecortex_results.json", "w"), indent=1)
print(json.dumps(A["baseline"], indent=1))
print(json.dumps(v, indent=1))
for r in runs:
    print(f"\n{r['arm']} seed {r['seed']}: talk ppl {r['talk_ppl']} read ppl {r['read_ppl']} R@1 {r['r_at_1']} ({r['seconds']}s)")
    for prompt, said in r["samples"].items():
        print(f"  > {prompt}")
        for s in said[:2]:
            print(f"      {s}")